# 📜 Tabular Classification

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MYTE21/Be.Fa.N/blob/rotom/notebooks/satire_as_fake_news.ipynb)

# 🌾 Rice Type Classification

Dataset for classification of rice by type.

## ⚙️ Imports

Import the necessary libraries and packages.

In [1]:
# Libraries: Standard
import platform
import warnings
warnings.filterwarnings("ignore")

# Libraries: External
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import  matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.optim import Adam
from torch.utils.data import Dataset, DataLoader

# Packages: External
from torchinfo import summary

# Packages: Local
from data import data

# Jupyter Magic Command
%matplotlib inline

<div style = "border: 2px solid #4B0082; padding: 5px 10px; border-radius: 10px;">
⚓️ <code style="color: green">import torch.nn as nn</code> imports the <code>neural network (nn)</code> module from PyTorch, which provides building blocks for <code>creating deep learning models</code>, such as <code>layers</code>, <code>activation functions</code>, and <code>loss functions</code>. ⤴️
</div>

<div style = "border: 2px solid #4B0082; padding: 5px 10px; border-radius: 10px;">
⚓️ <code style="color: green">from torchinfo import summary</code> prints a structured <code>summary of the model</code>, including the number of parameters and layer types. ⤴️
</div>

<div style = "border: 2px solid #4B0082; padding: 5px 10px; border-radius: 10px;">
⚓️ Since this is a <code style="color: green">binary classification problem</code>, accuracy is a sufficient metric. However, <code>for multi-class classification</code>, it is highly recommended to use <code>precision</code> and <code>recall</code>, as they provide better insights into specific issues that accuracy alone may not reveal. ⤴️
</div>

## 💻 Setting GPU

Unlike TensorFlow, PyTorch doesn't automatically detect the GPU, so we need to explicitly specify its availability.

<div style = "border: 2px solid #4B0082; padding: 5px 10px; border-radius: 10px;">
⚓️ Check PyTorch has access to <a href="https://pytorch.org/get-started/locally/">🔗 MPS (Metal Performance Shader</a>, Apple's GPU architecture). ⤵️
</div>

In [2]:
def get_mps() -> str:
    device_name = "mps" if torch.backends.mps.is_available() else "cpu"
    return device_name

<div style = "border: 2px solid #4B0082; padding: 5px 10px; border-radius: 10px;">
⚓️ Check PyTorch has access to <a href="https://developer.nvidia.com/cuda-zone">🔗 CUDA</a>, Nvidia (Windows). ⤵️
</div>

In [3]:
def get_cuda() -> str:
    device_name = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    return device_name

<div style = "border: 2px solid #4B0082; padding: 5px 10px; border-radius: 10px;">
⚓️ Set the device. ⤵️
</div>

In [4]:
device = get_mps() if platform.system() == "Darwin" else get_cuda()
print(f"Using device: {device}")

Using device: mps


## 📁 Data
An initial evaluation of the datasets to identify the basics of the data.

### 📂 Load Datasets

In [5]:
df = pd.read_csv(data.get_dataset_path("pytorch", "raw", "rice_type_classification", 2))

In [6]:
print("Rice type classification DataFrame: ")
df.head()

Rice type classification DataFrame: 


,id,Area,MajorAxisLength,MinorAxisLength,Eccentricity,ConvexArea,EquivDiameter,Extent,Perimeter,Roundness,AspectRation,Class
0,1,4537,92.229316,64.012769,0.719916,4677,76.004525,0.657536,273.085,0.764510,1.440796,1
1,2,2872,74.691881,51.400454,0.725553,3015,60.471018,0.713009,208.317,0.831658,1.453137,1
2,3,3048,76.293164,52.043491,0.731211,3132,62.296341,0.759153,210.012,0.868434,1.465950,1
3,4,3073,77.033628,51.928487,0.738639,3157,62.551300,0.783529,210.657,0.870203,1.483456,1
4,5,3693,85.124785,56.374021,0.749282,3802,68.571668,0.769375,230.332,0.874743,1.510000,1


## 🦊 Data Preprocessing